# `youtube_hd_downloader.py` Usage Examples

This notebook shows different ways to run the local downloader script from Jupyter.

Each example asks for the YouTube URL before running. Use only videos you have the right to access and download.

## 1) Setup

These helper cells keep commands readable and make sure we call the same Python interpreter that is running the notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

SCRIPT = Path('youtube_hd_downloader.py')

if not SCRIPT.exists():
    raise FileNotFoundError(f'Cannot find {SCRIPT.resolve()}')


In [ ]:
def ask_url(prompt='Enter YouTube URL: '):
    url = input(prompt).strip()
    if not url:
        raise ValueError('No URL provided. Please run the cell again and enter a URL.')
    return url


def run_downloader(*args):
    command = [sys.executable, str(SCRIPT), *args]
    print('Running:')
    print(' '.join(f'"{part}"' if ' ' in part else part for part in command))
    return subprocess.run(command, check=False)


## 2) Show Help

Run this when you want to see every available option.

In [ ]:
run_downloader('--help')


## 3) Basic 1080p+ Video Download

This downloads the best video/audio combination at 1080p or higher, then merges to MP4 when `ffmpeg` is available.

In [ ]:
url = ask_url()
run_downloader(url)


## 4) Save Into a Downloads Folder

`yt-dlp` output templates can use fields like `%(title)s`, `%(id)s`, and `%(ext)s`.

In [ ]:
url = ask_url()
run_downloader(
    '--output', 'downloads/%(title)s [%(id)s].%(ext)s',
    url,
)


## 5) Allow Lower Resolution Fallback

Use this when a video does not have a 1080p+ stream but you still want the best available version.

In [ ]:
url = ask_url()
run_downloader('--allow-lower-resolution', url)


## 6) Download Video With Manual Subtitles

This downloads the video plus manually provided English subtitles when the video has them.

In [ ]:
url = ask_url()
run_downloader('--write-subs', '--sub-langs', 'en', '--convert-subs', 'srt', url)


## 7) Download Video With Auto-Generated Subtitles

Use this when YouTube has automatic captions but no manually uploaded subtitle track.

In [ ]:
url = ask_url()
run_downloader('--write-auto-subs', '--sub-langs', 'en', '--convert-subs', 'srt', url)


## 8) Download Subtitles Only

This skips the video download and saves subtitle files only. The script prefers SRT and converts subtitles to SRT by default.

In [ ]:
url = ask_url()
run_downloader('--subtitles-only', '--write-auto-subs', '--sub-langs', 'en', '--convert-subs', 'srt', url)


## 9) Download Several Subtitle Languages

Pass a comma-separated language list. Use `all` if you want every available subtitle language.

In [ ]:
url = ask_url()
run_downloader(
    '--subtitles-only',
    '--write-subs',
    '--write-auto-subs',
    '--sub-langs', 'en,zh-Hans,ms',
    '--convert-subs', 'srt',
    url,
)


## 10) Choose Subtitle Format

Use `--sub-format` to prefer a downloaded subtitle format, and `--convert-subs srt` to make sure the saved file is SRT. The default download preference is `srt/vtt/best`.

In [ ]:
url = ask_url()
run_downloader(
    '--subtitles-only',
    '--write-auto-subs',
    '--sub-langs', 'en',
    '--sub-format', 'srt',
    '--convert-subs', 'srt',
    url,
)


## 11) Use Cookies for Login-Required Videos

By default the script looks for `cookies.txt` in the project root. You can also pass a custom cookies file path.

In [ ]:
url = ask_url()
run_downloader('--cookies', 'cookies.txt', '--write-auto-subs', '--convert-subs', 'srt', url)


## 12) Batch Example

Add your own URLs to the list, then run the same downloader settings for each one.

In [ ]:
raw_urls = input('Paste one or more YouTube URLs separated by commas: ')
urls = [url.strip() for url in raw_urls.split(',') if url.strip()]

if not urls:
    raise ValueError('No URLs provided. Please run the cell again and enter at least one URL.')

for video_url in urls:
    result = run_downloader(
        '--allow-lower-resolution',
        '--write-auto-subs',
        '--sub-langs', 'en',
        '--convert-subs', 'srt',
        '--output', 'downloads/%(title)s [%(id)s].%(ext)s',
        video_url,
    )
    if result.returncode != 0:
        print(f'Failed: {video_url}')
